# Rezervare Hotel cu Middleware pentru Membru Prioritar

Acest notebook demonstrează **middleware bazat pe funcții** folosind Microsoft Agent Framework. Construim pe exemplul fluxului de lucru condițional prin adăugarea unui strat de middleware care oferă membrilor prioritari privilegii speciale.

## Ce vei învăța:
1. **Middleware bazat pe funcții**: Interceptarea și modificarea rezultatelor funcțiilor
2. **Acces la context**: Citirea și modificarea `context.result` după execuție
3. **Implementarea logicii de afaceri**: Beneficii pentru membrii prioritari
4. **Suprascrierea rezultatului**: Schimbarea rezultatelor funcției în funcție de statutul utilizatorului
5. **Același flux de lucru, rezultate diferite**: Schimbări în comportament determinate de middleware

## Arhitectura fluxului de lucru cu Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Diferența crucială față de fluxul de lucru condițional:

**Fără Middleware** (14-conditional-workflow.ipynb):
- Paris nu are camere → Direcționează către alternative_agent

**Cu Middleware** (acest notebook):
- Utilizator normal + Paris → Fără camere → Direcționează către alternative_agent
- Utilizator prioritar + Paris → 🌟 Middleware suprascrie! → Disponibil → Direcționează către booking_agent

## Cerințe preliminare:
- Microsoft Agent Framework instalat
- Înțelegerea fluxurilor de lucru condiționale (vedeți 14-conditional-workflow.ipynb)
- Token GitHub sau cheie API OpenAI
- Înțelegere de bază a modelelor middleware


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Pasul 1: Definește modelele Pydantic pentru rezultate structurate

Aceste modele definesc **schema** pe care agenții o vor returna. Am adăugat un câmp `priority_override` pentru a urmări când middleware-ul modifică rezultatul disponibilității.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Pasul 2: Definirea bazei de date a membrilor cu prioritate

Pentru această demonstrație, vom simula o bază de date a membrilor cu prioritate. În producție, aceasta ar interoga o bază de date reală sau un API.

**Membri cu prioritate:**
- `alice@example.com` - membru VIP
- `bob@example.com` - membru Premium  
- `priority_user` - cont de testare


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## Pasul 3: Creează Instrumentul de Rezervare Hotel

La fel ca fluxul de lucru condițional, dar acum va fi interceptat de middleware!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Step 4: 🌟 Creează middleware-ul de verificare a priorității (FUNCȚIA CHEIE!)

Aceasta este **funcționalitatea de bază** a acestui notebook. Middleware-ul:

1. **Interceptează** apelul funcției hotel_booking
2. **Execută** funcția în mod normal apelând `next(context)`
3. **Inspectează** rezultatul în `context.result`
4. **Suprascrie** rezultatul dacă utilizatorul este prioritar și nu sunt camere disponibile
5. **Returnează** rezultatul modificat înapoi către agent

**Model cheie:**
```python
async def my_middleware(context, next):
    await next(context)  # Execută funcția
    # Acum context.result conține rezultatul funcției
    if some_condition:
        context.result = new_value  # Suprascrie!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## Pasul 5: Definirea Funcțiilor de Condiție pentru Rutare

Aceleași funcții de condiție ca în fluxul de lucru condițional - acestea inspectează rezultatul structurat pentru a determina rutarea.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## Pasul 6: Creează Executorul de Afișare Personalizat

Același executor ca înainte - afișează rezultatul final al fluxului de lucru.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Pasul 7: Încarcă Variabilele de Mediu

Configurează clientul LLM (Modele GitHub sau OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pasul 8: Crearea agenților AI cu Middleware

**DIFERENȚĂ CHEIE:** Când creăm availability_agent, transmitem parametrul `middleware`!

Așa introducem priority_check_middleware în fluxul de invocare a funcțiilor agentului.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Pasul 9: Construirea fluxului de lucru

Aceeași structură a fluxului de lucru ca înainte - rutare condiționată bazată pe disponibilitate.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Pasul 10: Cazul de test 1 - Utilizator obișnuit în Paris (Fără Suprascriere)

Un utilizator obișnuit încearcă să rezerve Paris → Nici o cameră → Redirecționează către alternative_agent


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Pasul 11: Cazul de test 2 - 🌟 Utilizator prioritar în Paris (CU Suprascriere!)

Un membru prioritar încearcă să rezerve Paris → Inițial, nu sunt camere → 🌟 Middleware suprascrie! → Direcționează către booking_agent

**Aceasta este demonstrația principală a puterii middleware-ului!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## Pasul 12: Caz de testare 3 - Utilizator prioritar în Stockholm (Deja disponibil)

Utilizator prioritar încearcă Stockholm → Camere disponibile → Nu este nevoie de suprascriere → Direcționează către booking_agent

Acest lucru arată că middleware-ul acționează doar când este nevoie!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## Concluzii Cheie și Concepte Middleware

### ✅ Ce ai învățat:

#### **1. Model Middleware bazat pe Funcții**

Middleware interceptează apelurile funcțiilor folosind o funcție async simplă:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # Înainte de execuția funcției
    print("Intercepting...")
    
    # Execută funcția
    await next(context)
    
    # După execuția funcției - inspectează rezultatul
    if context.result:
        # Modifică rezultatul dacă este necesar
        context.result = modified_value
```

#### **2. Acces la Context și Suprascrierea Rezultatului**

- `context.function` - Accesează funcția apelată
- `context.arguments` - Citește argumentele funcției
- `context.kwargs` - Accesează parametrii suplimentari
- `await next(context)` - Execută funcția
- `context.result` - Citește/modifică rezultatul funcției

#### **3. Implementarea Logicii de Afaceri**

Middleware-ul nostru implementează beneficii pentru membri cu prioritate:
- **Utilizatori obișnuiți**: Fără modificări, flux standard
- **Utilizatori cu prioritate**: Suprascrie „fără disponibilitate” → „disponibil”
- **Logică condiționată**: Suprascrie doar când este necesar

#### **4. Același Flux de Lucru, Rezultate Diferite**

Puterea middleware-ului:
- ✅ Nicio modificare a structurii fluxului de lucru
- ✅ Nicio modificare a funcției instrumentului
- ✅ Nicio modificare a logicii de rutare condiționată
- ✅ Doar middleware → Comportament diferit!

### 🚀 Aplicații în Lumea Reală:

1. **Funcționalități VIP/Premium**
   - Suprascrie limitele de rată pentru utilizatorii premium
   - Oferă acces prioritar la resurse
   - Deblochează funcționalități premium dinamic

2. **Testare A/B**
   - Direcționează utilizatorii către implementări diferite
   - Testează funcționalități noi cu utilizatori specifici
   - Lansări treptate de funcționalități

3. **Securitate & Conformitate**
   - Auditează apelurile funcțiilor
   - Blochează operațiuni sensibile
   - Aplică reguli de afaceri

4. **Optimizarea Performanței**
   - Cachează rezultatele pentru utilizatori specifici
   - Sarce operațiunile costisitoare când este posibil
   - Alocare dinamică a resurselor

5. **Gestionarea Erorilor & Reîncercări**
   - Prinde și gestionează erorile cu grație
   - Implementează logica de retry
   - Revine la implementări alternative

6. **Înregistrare & Monitorizare**
   - Urmărește timpii de execuție ai funcțiilor
   - Înregistrează parametrii și rezultatele
   - Monitorizează tiparele de utilizare

### 🔑 Diferențe Cheie față de Decoratori:

| Caracteristică | Decorator | Middleware |
|---------|-----------|------------|
| **Scop** | Funcție unică | Toate funcțiile din agent |
| **Flexibilitate** | Fixă la definire | Dinamică în execuție |
| **Context** | Limitat | Context complet al agentului |
| **Compoziție** | Mai mulți decoratori | Conductă de middleware |
| **Agent-Aware** | Nu | Da (acces la starea agentului) |

### 📚 Când să folosești Middleware:

✅ **Folosește middleware când:**
- Ai nevoie să modifici comportamentul în funcție de starea utilizatorului/sesiunii
- Vrei să aplici logică la mai multe funcții
- Ai nevoie de acces la contextul la nivel de agent
- Implementezi preocupări transversale (logging, autentificare etc.)

❌ **Nu folosi middleware când:**
- Validezi simplu input (folosește Pydantic)
- Logică specifică funcției (păstrează în funcție)
- Modificări unice (schimbă doar funcția)

### 🎓 Modele Avansate:

```python
# Mai multe middleware-uri (ordinea de execuție contează!)
middleware=[
    logging_middleware,      # Înregistrează primul
    auth_middleware,         # Apoi verifică autentificarea
    cache_middleware,        # Apoi verifică cache-ul
    rate_limit_middleware,   # Apoi limitează rata
    priority_check_middleware  # În cele din urmă verificarea priorității
]

# Execuție condiționată a middleware-ului
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # Modifică rezultatul
    else:
        # Sară complet peste execuție
        context.result = cached_value
```

### 🔗 Concepte Asociate:

- **Agent Middleware**: Interceptează apelurile agent.run()
- **Function Middleware**: Interceptează apelurile funcțiilor instrument (ce am folosit noi!)
- **Middleware Pipeline**: Lanț de middleware care se execută în ordine
- **Propagarea Contextului**: Transmite starea prin lanțul de middleware-uri


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
